# LoRA Adapter Rollouts

**LoRA (Low-Rank Adaptation)** is a technique for fine-tuning large language
models by training a small set of adapter weights instead of modifying the
full model. A LoRA adapter is typically only 0.1-1% of the base model size,
making it cheap to train, store, and swap.

In production, you often need to deploy new LoRA adapter versions without
downtime. **LoRA rollouts** let you:

- Deploy a new adapter version alongside the existing one
- Gradually shift traffic from the old adapter to the new one
- Monitor latency and error rates during the transition
- Roll back instantly if issues are detected

This notebook covers deploying a base model with LoRA adapters,
configuring cache-aware LoRA routing in the EPP, performing a
gradual traffic rollout, and rolling back if needed.

In [ ]:
# Environment setup
import os

os.environ["NAMESPACE"] = "llm-d"
os.environ["BASE_MODEL"] = "Qwen/Qwen3-32B"
os.environ["LORA_V1"] = "my-org/qwen3-32b-lora-v1"
os.environ["LORA_V2"] = "my-org/qwen3-32b-lora-v2"

print("Namespace:", os.environ["NAMESPACE"])
print("Base model:", os.environ["BASE_MODEL"])
print("LoRA v1:", os.environ["LORA_V1"])
print("LoRA v2:", os.environ["LORA_V2"])

## How LoRA Works with vLLM

vLLM supports serving multiple LoRA adapters on a single base model.
The base model weights are loaded once, and LoRA adapter weights are
loaded on top as small delta matrices.

```
Base Model (Qwen3-32B)
+--------------------+
|  Attention layers   |  Shared across all adapters
|  FFN layers         |
+--------------------+
        |
   +----+----+
   |         |
   v         v
+------+  +------+
| LoRA |  | LoRA |   Small delta matrices (~50 MB each)
|  v1  |  |  v2  |   Applied at inference time
+------+  +------+
```

Clients specify which adapter to use via the `model` field in the
request. The EPP routes requests to replicas that have the requested
adapter loaded, preferring those with cached prefixes for that adapter.

In [ ]:
# Deploy the base model with the initial LoRA adapter (v1)

model_server = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: vllm
  namespace: llm-d
spec:
  replicas: 2
  selector:
    matchLabels:
      app: vllm
  template:
    metadata:
      labels:
        app: vllm
    spec:
      containers:
        - name: vllm
          image: vllm/vllm-openai:latest
          args:
            - --model
            - Qwen/Qwen3-32B
            - --enable-lora
            - --lora-modules
            - '{"name": "qwen3-lora-v1", "path": "/lora-adapters/v1", "base_model_name": "Qwen/Qwen3-32B"}'
            - --max-loras
            - "4"
            - --max-lora-rank
            - "64"
            - --enable-prefix-caching
          resources:
            limits:
              nvidia.com/gpu: 1
          volumeMounts:
            - name: lora-adapters
              mountPath: /lora-adapters
      volumes:
        - name: lora-adapters
          persistentVolumeClaim:
            claimName: lora-adapters-pvc
"""

with open("/tmp/vllm-lora.yaml", "w") as f:
    f.write(model_server)

!kubectl apply -f /tmp/vllm-lora.yaml

print("Model server deployed with LoRA v1 adapter.")
print("Waiting for pods...")
!kubectl wait --for=condition=ready pod -l app=vllm -n $NAMESPACE --timeout=600s
print("Model server pods are ready.")
print()
print("Key vLLM LoRA flags:")
print("  --enable-lora:     Activates LoRA adapter support")
print("  --lora-modules:    Specifies adapter name, path, and base model")
print("  --max-loras:       Maximum adapters loaded simultaneously (4)")
print("  --max-lora-rank:   Maximum LoRA rank supported (64)")

In [ ]:
# Configure cache-aware LoRA routing in the EPP
# The EPP needs to know which replicas have which adapters loaded
# and route requests accordingly

epp_config = """apiVersion: v1
kind: ConfigMap
metadata:
  name: epp-config
  namespace: llm-d
data:
  config.yaml: |
    scorers:
      - name: lora-affinity
        type: lora-affinity
        weight: 0.4
        config:
          # Prefer replicas that already have the requested adapter loaded
          # This avoids the latency of loading an adapter on-the-fly
          adapterLoadPenalty: 0.5
      - name: prefix-cache
        type: prefix-cache
        weight: 0.3
      - name: load-aware
        type: load-aware
        weight: 0.3
"""

with open("/tmp/epp-lora-config.yaml", "w") as f:
    f.write(epp_config)

!kubectl apply -f /tmp/epp-lora-config.yaml

# Restart EPP to pick up new config
!kubectl rollout restart deployment epp -n $NAMESPACE
!kubectl wait --for=condition=ready pod -l app=epp -n $NAMESPACE --timeout=120s

print("EPP configured with LoRA-affinity scoring.")
print()
print("Scoring pipeline:")
print("  1. lora-affinity (40%): Prefer replicas with the adapter already loaded")
print("  2. prefix-cache (30%):  Prefer replicas with matching prefix cache")
print("  3. load-aware (30%):    Balance load across replicas")

In [ ]:
# Send requests using the LoRA v1 adapter
import subprocess
import json

GATEWAY_IP = subprocess.check_output(
    "kubectl get svc envoy-gateway -n llm-d "
    "-o jsonpath='{.status.loadBalancer.ingress[0].ip}'",
    shell=True
).decode().strip().strip("'")

# Request using the LoRA adapter name as the model
result = subprocess.run(
    ["curl", "-s", f"http://{GATEWAY_IP}:8080/v1/chat/completions",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({
         "model": "qwen3-lora-v1",
         "messages": [{"role": "user", "content": "Summarize the benefits of LoRA fine-tuning."}],
         "max_tokens": 150,
     })],
    capture_output=True, text=True
)

response = json.loads(result.stdout)
if "choices" in response:
    print("LoRA v1 response:")
    print(response["choices"][0]["message"]["content"][:300])
    print()
    print(f"Model: {response.get('model', 'N/A')}")
else:
    print(json.dumps(response, indent=2))

In [ ]:
# Deploy a second LoRA adapter version (v2)
# We add the new adapter to the existing model server without restarting it
# vLLM supports dynamic adapter loading via the API

# Register the new adapter with each vLLM instance
vllm_pods = subprocess.check_output(
    "kubectl get pods -n llm-d -l app=vllm -o jsonpath='{.items[*].metadata.name}'",
    shell=True
).decode().strip().strip("'").split()

print(f"Registering LoRA v2 adapter on {len(vllm_pods)} pods...")

for pod in vllm_pods:
    # Use vLLM's adapter loading API
    result = subprocess.run(
        ["kubectl", "exec", "-n", "llm-d", pod, "--",
         "curl", "-s", "-X", "POST", "http://localhost:8000/v1/load_lora_adapter",
         "-H", "Content-Type: application/json",
         "-d", json.dumps({
             "lora_name": "qwen3-lora-v2",
             "lora_path": "/lora-adapters/v2",
         })],
        capture_output=True, text=True
    )
    print(f"  {pod}: {result.stdout.strip()}")

print()
print("LoRA v2 adapter loaded on all replicas.")
print("Both v1 and v2 are now available for serving.")

In [ ]:
# Configure traffic splitting for gradual rollout
# Phase 1: 90% v1, 10% v2

import time
import threading
import random

def send_to_adapter(adapter_name, prompt):
    """Send a request to a specific LoRA adapter."""
    result = subprocess.run(
        ["curl", "-s", "-w", "\n%{time_starttransfer} %{http_code}",
         f"http://{GATEWAY_IP}:8080/v1/chat/completions",
         "-H", "Content-Type: application/json",
         "-d", json.dumps({
             "model": adapter_name,
             "messages": [{"role": "user", "content": prompt}],
             "max_tokens": 50,
         })],
        capture_output=True, text=True
    )
    lines = result.stdout.strip().split("\n")
    timing = lines[-1].split() if len(lines) > 1 else ["0", "0"]
    return {
        "ttft": float(timing[0]),
        "status_code": timing[1] if len(timing) > 1 else "0",
    }

def run_rollout_phase(v1_pct, v2_pct, num_requests=50):
    """Run traffic with the given split and collect metrics."""
    v1_results = []
    v2_results = []

    for _ in range(num_requests):
        roll = random.randint(1, 100)
        if roll <= v1_pct:
            r = send_to_adapter("qwen3-lora-v1", "What is machine learning?")
            v1_results.append(r)
        else:
            r = send_to_adapter("qwen3-lora-v2", "What is machine learning?")
            v2_results.append(r)

    return v1_results, v2_results

def report_results(name, results):
    if not results:
        print(f"  {name}: no requests")
        return
    ttfts = [r["ttft"] * 1000 for r in results]
    errors = sum(1 for r in results if r["status_code"] != "200")
    print(f"  {name}: {len(results)} requests, "
          f"avg TTFT={sum(ttfts)/len(ttfts):.0f}ms, "
          f"errors={errors}")

# Phase 1: 90/10
print("=== Phase 1: 90% v1 / 10% v2 ===")
v1_r, v2_r = run_rollout_phase(90, 10, num_requests=50)
report_results("v1", v1_r)
report_results("v2", v2_r)

print()
time.sleep(5)

# Phase 2: 50/50
print("=== Phase 2: 50% v1 / 50% v2 ===")
v1_r, v2_r = run_rollout_phase(50, 50, num_requests=50)
report_results("v1", v1_r)
report_results("v2", v2_r)

print()
time.sleep(5)

# Phase 3: 0/100
print("=== Phase 3: 0% v1 / 100% v2 ===")
v1_r, v2_r = run_rollout_phase(0, 100, num_requests=50)
report_results("v1", v1_r)
report_results("v2", v2_r)

print()
print("Rollout complete. All traffic is now served by LoRA v2.")

In [ ]:
# Monitor latency and error rates during rollout

print("=== Rollout Health Metrics ===")
print()

# Per-adapter latency
print("TTFT by adapter:")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=histogram_quantile(0.99,rate(epp_request_ttft_seconds_bucket{namespace="llm-d"}[5m]))' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    model = r['metric'].get('model', 'unknown')
    val = float(r['value'][1]) * 1000
    print(f'  {model} p99 TTFT: {val:.0f} ms')
" 2>/dev/null || echo "  (metrics not yet available)"

print()

# Error rates by adapter
print("Error rates by adapter:")
!kubectl exec -n monitoring deploy/kube-prometheus-prometheus -- \
    wget -qO- 'http://localhost:9090/api/v1/query?query=sum(rate(epp_requests_total{namespace="llm-d",status="error"}[5m]))by(model)' \
    2>/dev/null | python3 -c "
import json, sys
data = json.load(sys.stdin)
for r in data.get('data', {}).get('result', []):
    model = r['metric'].get('model', 'unknown')
    rate = float(r['value'][1])
    print(f'  {model}: {rate:.4f} errors/sec')
" 2>/dev/null || echo "  (error metrics not yet available)"

print()
print("Things to watch during a rollout:")
print("  - Error rate spikes on the new adapter version")
print("  - TTFT increase (may indicate adapter loading delays)")
print("  - Quality degradation (requires offline evaluation)")

In [ ]:
# Rollback procedure: shift all traffic back to v1

print("=== Rollback Procedure ===")
print()
print("If issues are detected during rollout, shift traffic back to v1:")
print()
print("Option 1: Client-side rollback")
print("  Change the 'model' field in requests back to 'qwen3-lora-v1'")
print("  This is instant and requires no infrastructure changes.")
print()
print("Option 2: Unload the problematic adapter")

# Demonstrate unloading an adapter
print("  Unloading qwen3-lora-v2 from all replicas...")

vllm_pods = subprocess.check_output(
    "kubectl get pods -n llm-d -l app=vllm -o jsonpath='{.items[*].metadata.name}'",
    shell=True
).decode().strip().strip("'").split()

for pod in vllm_pods:
    result = subprocess.run(
        ["kubectl", "exec", "-n", "llm-d", pod, "--",
         "curl", "-s", "-X", "POST", "http://localhost:8000/v1/unload_lora_adapter",
         "-H", "Content-Type: application/json",
         "-d", json.dumps({"lora_name": "qwen3-lora-v2"})],
        capture_output=True, text=True
    )
    print(f"    {pod}: {result.stdout.strip()}")

print()
print("  Rollback complete. Only qwen3-lora-v1 is now available.")
print("  Any requests for qwen3-lora-v2 will return an error,")
print("  prompting clients to switch back to v1.")

## Summary

LoRA adapter rollouts allow you to deploy, test, and migrate between
fine-tuned model versions without downtime or base model reloading.

### Rollout Workflow

1. Deploy base model with initial adapter (v1)
2. Load new adapter (v2) dynamically via the vLLM API
3. Gradually shift traffic: 90/10 -> 50/50 -> 0/100
4. Monitor latency and errors at each phase
5. Roll back by unloading v2 or redirecting traffic if issues arise

### Key Configuration

- vLLM: `--enable-lora --max-loras 4 --max-lora-rank 64`
- EPP: `lora-affinity` scorer routes to replicas with the adapter loaded
- Clients specify the adapter via the `model` field in the request

### Best Practices

- Always run a canary phase (90/10) before full rollout
- Monitor both latency and quality metrics during rollout
- Keep the old adapter loaded during rollout for instant rollback
- Use adapter naming conventions that include version numbers
- Test the rollback procedure before you need it

### Next Steps

- **Flow Control**: Use priority bands to protect LoRA rollout traffic
  from interfering with production workloads.
- **Autoscaling**: Configure HPA to scale based on per-adapter queue depth.